# Tutorial: how to use `pangeo-fish`

**Overview.**

This notebook demonstrates fish geolocation with `pangeo-fish`, using biologging data from the study by M. Gonze et al., "Combining acoustic telemetry with archival tagging to investigate the spatial dynamics of the understudied pollack *Pollachius pollachius*".

We use tag `A19124`, which includes Data Storage Tag (DST) observations, acoustic detections, and release and recapture/death information. The reference ocean temperatures come from Copernicus Marine reanalysis data; the configuration below selects IBI and documents GLORYS12 as an alternative.

Run this notebook with the current ETN workshop image, `ghcr.io/tinaok/pangeo-fish-notebook:2026.06.04-3c50042`, using the **Python (pangeo-fish)** kernel; see [Getting started](users-getting-started.md). The Docker image provides the required packages, including video support.

Outputs from the previous GLORYS run have been cleared. Run the notebook in the workshop environment to generate results for the selected configuration.

**Workflow.**

Let's first summarize the key steps for running the geolocation:

1. **Define the configuration:** define the required parameters for the analysis.
2. **Compare the reference data with the DST information:** compare the data from the reference model with the biologging data. 
3. **Regrid the comparison to HEALPix:** resample the comparison onto a hierarchical equal-area grid.
4. **Construct the temporal emission matrix:** convert temperature differences into emission likelihoods on the HEALPix grid.
5. **Construct another emission matrix with the acoustic detections:** construct spatial constraints from the acoustic detections.
6. **Combine and normalize the matrices:** merge and normalize the two _pdfs_.
7. **Estimate (or _fit_) the geolocation model:** determine the parameters of the model based on the normalized emission matrix.
8. **Compute the state probabilities and generate trajectories:** compute the fish's location probability distribution and generate subsequent trajectories.
9. **Visualization:** visualize the evolution of the spatial probabilities over time and export the video.

Throughout this tutorial, you will gain practical experience in setting up and executing a typical workflow using `pangeo-fish` such that you can then apply the tool to your own study.

## 1. Initialization and configuration

We import the required libraries, define parameters for the workflow, and configure a local Dask cluster. The workshop Docker image already provides the dependencies.

The following cell disables CUDA for this session to use the CPU configuration.

In [ ]:
# Use CPU execution.
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""

### Required libraries

In [ ]:
from pint_xarray import unit_registry as ureg
import hvplot.xarray
import xarray as xr
import sys

# sys.path.append("../")
import pangeo_fish
from pangeo_fish.xarray_update import Accessor
from pangeo_fish.helpers import multiplot_healpix, multi_map_with_synced_sliders

### Parameters

In [ ]:
# SECTION 1 — Initialization and configuration
# DST identifier; also used to organize outputs throughout the workflow.
tag_name = "A18828"
tag_name = "A19124"

# Root URL for tag data; a local path can also be used.
tag_root = "https://data-taos.ifremer.fr/demo/tag/tag/"

# Output location and filesystem options, used throughout the workflow.
# Use local storage for this workshop.
scratch_root = "./test"
storage_options = None

# Time chunk size, used throughout loading and processing.
# 1 = one time step per chunk; -1 = all time steps in one chunk.
chunk_time = 1

# SECTION 2 — Compare reference ocean data with DST observations
# Select the Copernicus Marine ocean model; keep only one assignment active.
# Both options are supported by the workshop image specified in Section 2.

# IBI Reanalysis — Regional model, ~3 km, daily (default)
# https://data.marine.copernicus.eu/product/IBI_MULTIYEAR_PHY_005_002
copernicus_model_name = "cmems_mod_ibi_phy-temp_my_0.027deg_P1D-m"

# GLORYS12 Reanalysis — Global model, ~9 km, daily
# https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030
# copernicus_model_name = "cmems_mod_glo_phy_my_0.083deg_P1D-m"

# None selects direct Copernicus Marine loading with copernicus_model_name.
# Alternatively, supply an intake .yaml catalog or a kerchunk .parq/.json reference.
ref_url = None

# Geographic bounds for model loading; also used in Sections 3 and 9.
bbox = {"latitude": [46, 51.5], "longitude": [-7, 0]}
# Reject cells whose model seabed is shallower than this fraction of the
# maximum observed tag depth in a model time bin; 0 disables this filter.
relative_depth_threshold = 0

# SECTION 3 — HEALPix regridding
# Spatial dimension used for reductions and HMM calculations in Sections 4–8.
dims = ["cells"]

# HEALPix resolution (approximate equivalent pixel size)
# Each refinement level divides the cell area by 4.
# Total number of cells = 12 * 4**refinement_level
# Equivalent size = sqrt(cell area) on a spherical Earth of radius 6371 km,
# not an exact geometric edge length. Also used by later HEALPix plots.
#
# refinement_level = 10  # ~6.4 km
refinement_level = 11    # ~3.2 km (similar to the 3 km IBI model)
# refinement_level = 12  # ~1.6 km
#
# Higher refinement levels provide finer spatial resolution
# but increase the number of cells and computational cost.

# Earth model for HEALPix plots in Sections 3, 4, and 6.
# "sphere" matches the regridding helper's default; "WGS84" is ellipsoidal.
ellipsoid = "sphere"

# SECTION 4 — Temperature-based emission probability
# Standard deviation of the Gaussian temperature-difference likelihood, in °C.
differences_std = 0.75
# Angular Gaussian widths for release and recapture/death locations, in radians.
initial_std = 1e-4
recapture_std = 1e-4

# SECTION 5 — Acoustic detection probability
# Maximum detection distance assumed around each receiver.
receiver_buffer = ureg.Quantity(2000, "m")

# SECTION 7 — Optimization of movement parameters
# Earth radius used to convert movement distances to angular distances.
earth_radius = ureg.Quantity(6371, "km")
# Speed scale used to bound the search for the movement parameter sigma.
maximum_speed = ureg.Quantity(60, "km / day")
# Scale the upper search bound:
# max_sigma = maximum_speed * Δt * adjustment_factor / earth_radius (radians).
# Δt comes from the emission time coordinate, not the video time_step below.
# For daily data, 1 gives an upper bound of ~0.0094 radians (60 km).
adjustment_factor = 1
# Gaussian truncation factor for HealpixConv; unused by LargeHealpixConv.
truncate = 4
# Search tolerance; smaller values may require more objective evaluations.
tolerance = 1e-6

# SECTION 8 — State probabilities and trajectories
# Trajectory summaries to compute; also used for plotting in Section 9.
track_modes = ["mean", "mode"]
# Additional quantities to compute from trajectories using MovingPandas.
additional_track_quantities = ["speed", "distance"]

# SECTION 9 — Visualization and video generation
# Number of data time steps between video frames
# 1 = include every time step
# 3 = include every third time step
# See Section 9 for the pinned renderer's final-time-step limitation.
time_step = 1

In [ ]:
# Define target root directories for storing analysis results.
target_root = f"{scratch_root}/{tag_name}"

# Defines default chunk size for optimization.
default_chunk = {"time": chunk_time, "lat": -1, "lon": -1}
default_chunk_dims = {"time": chunk_time}
default_chunk_dims.update({d: -1 for d in dims})

### Dask

Dask runs independent calculations in parallel and manages data in chunks.

In [ ]:
# Set up a local cluster for distributed computing.
from distributed import LocalCluster

cluster = LocalCluster()
client = cluster.get_client()
client

### Opening the biologging data
Now that everything is set up, we can start by loading the biologging data (or _tag_)

In [ ]:
from pangeo_fish.helpers import load_tag

tag, tag_log, time_slice = load_tag(
    tag_root=tag_root, tag_name=tag_name, storage_options=storage_options
)
tag

You can plot the time series of the DST with the function `plot_tag()`:

In [ ]:
from pangeo_fish.helpers import plot_tag

plot = plot_tag(
    tag=tag,
    tag_log=tag_log,
    # you can directly save the plot if you want
    save_html=True,
    storage_options=storage_options,
    target_root=target_root,
)
plot

The red points are the DST observations used in the following steps, restricted to the period when the tag is assumed to have been attached to a living fish.

### Credentials for reference model
Reuse the credentials saved by the Copernicus Marine Toolbox. The cell below checks its default credentials file (`~/.copernicusmarine/.copernicusmarine-credentials`). If it is missing, `copernicusmarine.login()` prompts for your Copernicus Marine username and password and saves them for future sessions.

Run this cell before opening the reference model. You do not need to create or read a separate `copernicus_credentials.txt` file, or pass `Username` and `Password` to `load_model()`: the Toolbox reads its saved credentials automatically.

If saved credentials are rejected, run `copernicusmarine.login(force_overwrite=True)` to enter and save them again.

See the [Copernicus Marine credentials guide](https://help.marine.copernicus.eu/en/articles/8185007-copernicus-marine-toolbox-credentials-configuration) for details.

In [ ]:
from pathlib import Path

import copernicusmarine

credentials_file = Path.home() / ".copernicusmarine" / ".copernicusmarine-credentials"

if credentials_file.is_file():
    print("Using saved Copernicus Marine credentials.")
else:
    copernicusmarine.login()  # Prompts for username and password when needed.


## 2. Compare reference ocean data with DST observations

### Open the reference ocean model

With `ref_url = None`, `load_model()` uses `copernicus_model_name`. To switch between IBI and GLORYS12, comment out the active assignment in the parameters cell and uncomment the other one.

Both dataset IDs are listed by Copernicus Marine ([IBI](https://data.marine.copernicus.eu/product/IBI_MULTIYEAR_PHY_005_002/services), [GLORYS12](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030/services)) and supported by the ETN workshop image `ghcr.io/tinaok/pangeo-fish-notebook:2026.06.04-3c50042`. This image uses the [tinaok fork at commit `3c50042`](https://github.com/tinaok/pangeo-fish/tree/3c50042), whose loader handles IBI's separate temperature, sea-level, and static datasets internally. No additional notebook configuration is needed; IBI is the default.

In [ ]:
from pangeo_fish.helpers import load_model, compute_diff

reference_model = load_model(
    uri=ref_url,
    tag_log=tag_log,
    time_slice=time_slice,
    bbox=(bbox | {"max_depth": tag_log["pressure"].max()}),
    chunk_time=chunk_time,
    remote_options={},
    copernicus_model_name=copernicus_model_name,
)

### Compute temperature differences

Compare DST temperatures with model temperatures at the observed depths, grouped into the model's time bins. The resulting field describes temperature differences at each candidate location and time.

In [ ]:
%%time
diff = compute_diff(
    reference_model=reference_model,
    tag_log=tag_log,
    relative_depth_threshold=relative_depth_threshold,
    chunk_time=chunk_time,
)[0]
diff = diff.compute()
diff

### Check data availability over time

The following plot counts non-NaN spatial grid cells at each time step. This is an important quality check before running the Hidden Markov Model (HMM).

Counts may vary over time, but a sudden drop can indicate missing data or a processing issue. A count of zero means that no valid spatial values are available at that time step, which can prevent the HMM from running correctly. Investigate these gaps before continuing.

`.count()` includes zero-valued cells; it does not sum their values or check that they are positive.

In [ ]:
%%time
diff["diff"].count(["lat", "lon"]).plot()

In [ ]:
%%time
diff.to_zarr(
    f"{target_root}/diff.zarr", mode="w", storage_options=storage_options, zarr_format=2
)

## 3. HEALPix regridding

HEALPix (Hierarchical Equal Area isoLatitude Pixelization) provides a hierarchical equal-area grid that supports multiresolution analysis and efficient spatial indexing. Equal area does not mean that cell shapes are identical or that all spatial distortion is eliminated; see the [HEALPix documentation](https://healpix.sourceforge.io/).

The `pangeo-fish` regridding helper constructs the target grid and resamples the temperature-difference field onto its cells.

In [ ]:
from pangeo_fish.helpers import (
    open_diff_dataset,
    regrid_dataset_hpresample,
)

In [ ]:
# Open the previous dataset (only necessary if you resume the notebook from here)
diff = open_diff_dataset(target_root=target_root, storage_options=storage_options)
diff

In [ ]:
reshaped, resampling_info = regrid_dataset_hpresample(
    ds=diff, refinement_level=refinement_level, dims=dims
)
reshaped

In [ ]:
resampling_info["bbox"] = bbox
import json

# Save the regridding metadata and geographic bounds alongside the results.
with open(f"{target_root}/resampling_info.json", "w") as f:
    json.dump(resampling_info, f)

with open(f"{target_root}/resampling_info.json") as f:
    resampling_info = json.load(f)
resampling_info

Repeat the data-availability check after regridding. Counts may differ because the spatial resolution and grid coverage have changed, but they should not drop to zero. Investigate sudden drops before continuing.

In [ ]:
reshaped["diff"].count(dims).plot()

In [ ]:
# Saves the result
reshaped.chunk(default_chunk_dims).to_zarr(
    f"{target_root}/diff-regridded.zarr",
    mode="w",
    consolidated=True,
    compute=True,
    storage_options=storage_options,
    zarr_format=2,
)

The following map shows the temperature-difference field on the HEALPix grid. Inspect its spatial coverage and how it changes over time.

In [ ]:
reshaped.healpix.plot(
    var=["diff", "H0"], refinement_level=refinement_level, alpha=0.8, ellipsoid=ellipsoid
)

## 4. Temperature-based emission probability

We convert the temperature differences into Gaussian emission likelihoods, centered on zero with standard deviation `differences_std`. Smaller differences give higher likelihoods. Release and recapture/death locations provide additional spatial information.

Although the variable is named `emission_pdf`, its `pdf` field is not yet normalized over spatial cells. The combined emission distribution is normalized in Section 6.

In [ ]:
from pangeo_fish.helpers import compute_emission_pdf

In [ ]:
# Open the previous dataset (only necessary if you resume the notebook from here)
differences = xr.open_dataset(
    f"{target_root}/diff-regridded.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)

# ... and compute the emission matrices
emission_pdf = compute_emission_pdf(
    diff_ds=differences,
    events_ds=tag["tagging_events"].ds,
    differences_std=differences_std,
    initial_std=initial_std,
    recapture_std=recapture_std,
    dims=dims,
    chunk_time=chunk_time,
)[0]
emission_pdf

Plot the temperature-based emission likelihoods to inspect their spatial pattern.

In [ ]:
emission_pdf.healpix.plot(["initial", "final", "pdf"], refinement_level, ellipsoid=ellipsoid)

Repeat the data-availability check for `emission_pdf["pdf"]`. `.count(dims)` counts non-missing values, including zeros, so a positive count alone does not rule out zero total likelihood.

The additional `.sum(dims, min_count=1)` diagnostic checks the total likelihood. It need not equal 1 before normalization, but zero total likelihood is problematic. `min_count=1` keeps an entirely missing time step as NaN instead of reporting a sum of zero. Investigate zero or missing totals before continuing.

In [ ]:
emission_pdf = emission_pdf.chunk(default_chunk_dims).persist()
emission_pdf["pdf"].count(dims).plot()

In [ ]:
emission_pdf["pdf"].sum(dims, min_count=1).rename("Total temperature likelihood").plot()

In [ ]:
# Save the dataset
emission_pdf.to_zarr(
    f"{target_root}/emission_diff.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

In [ ]:
emission_pdf.attrs["sources"] = emission_pdf.attrs.get("sources", []) + [
    "temperature"
]  # Record the evidence sources included so far.
emission_pdf.to_zarr(
    f"{target_root}/total_pdf.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

## 5. Acoustic detection probability

We construct another emission distribution from acoustic detections. **This step requires acoustic detections and receiver locations in the tag data.** These detections provide spatial constraints that complement the temperature likelihoods. We combine the distributions and normalize the result in Section 6.

In [ ]:
from pangeo_fish.helpers import compute_acoustic_pdf

In [ ]:
# Load the previous emission pdf and compute the emission probabilities based on acoustic detections
emission_diff = xr.open_dataset(
    f"{target_root}/emission_diff.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
acoustic_pdf = compute_acoustic_pdf(
    emission_ds=emission_diff,
    tag=tag,
    receiver_buffer=receiver_buffer,
    chunk_time=chunk_time,
    dims=dims,
)[0]
acoustic_pdf = acoustic_pdf.persist()
acoustic_pdf

The following plot overlays detection times with the number of cells satisfying `acoustic != 0`. This differs from the earlier non-missing-cell counts. Note that `NaN != 0` is also true, so missing cells can contribute to this diagnostic; it is not a strict count of positive probabilities.

In [ ]:
tag["acoustic"]["deployment_id"].hvplot.scatter(c="red", marker="x") * (
    acoustic_pdf["acoustic"] != 0
).sum(dim=dims).hvplot()

At acoustic detection times, the number of possible grid cells can decrease sharply. These cells correspond to the area around the receivers where the fish was detected, under the assumed detection range. Interpret the plotted count with the missing-value caveat above.

These detections provide strong spatial constraints for the geolocation model.

The next cell optionally saves the acoustic emission distribution separately. The subsequent merge uses the in-memory dataset.

In [ ]:
acoustic_pdf.to_zarr(
    f"{target_root}/acoustic.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
)

### Merge the evidence sources

In [ ]:
# Open the currently saved evidence sources.
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
current_total

In [ ]:
if "acoustic" in current_total.data_vars:
    pass
else:
    current_total = current_total.load()
    current_total.attrs["sources"] = current_total.attrs.get("sources", []) + [
        "acoustic"
    ]

    merged_pdf = current_total.merge(
        acoustic_pdf.drop_indexes(["cell_ids"]), compat="override"
    )

    merged_pdf.to_zarr(
        f"{target_root}/total_pdf.zarr",
        mode="w",
        consolidated=True,
        storage_options=storage_options,
        zarr_format=2,
    )
merged_pdf

## 6. Combine and normalize the emission distributions

`normalize_pdf()` multiplies the temperature and acoustic evidence, incorporates the final-location distribution when available, and normalizes the combined `pdf` over spatial cells at each time step. The `initial`, `final`, and `mask` fields are retained separately.

In [ ]:
from pangeo_fish.helpers import combine_pdfs
from pangeo_fish.helpers import normalize_pdf

In [ ]:
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
current_total

In [ ]:
emission_pdf = normalize_pdf(
    ds=current_total,
    chunks=default_chunk_dims,
    dims=dims,
    exclude=("initial", "final", "mask"),
    excluded_pdf=None,  # ("acoustic",)
)[0]

emission_pdf.to_zarr(
    f"{target_root}/combined.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)
combined = emission_pdf

(
    emission_pdf.sum(dims).hvplot(
        width=500, ylim=(0, 2), title="Spatial sums by variable"
    )
    + emission_pdf.count(dims).hvplot(
        width=500, title="Number of non-missing grid cells"
    )
).opts(shared_axes=False)

For the normalized `pdf` field, `.sum(dims)` should be approximately 1 at every time step. The other retained fields are not all normalized spatial probabilities: for example, summing `mask` counts cells in the mask.

The `.count(dims)` panel checks data availability and includes zeros. Investigate missing or zero `pdf` counts or totals before optimization; an all-NaN field also produces zero with the default `.sum()` behavior.

In [ ]:
combined.healpix.plot("pdf", refinement_level=refinement_level, ellipsoid=ellipsoid)

## 7. Optimization of movement parameters

We estimate `sigma`, the angular width of the movement kernel between successive model time steps, using the normalized emission distribution.

1. Define the lower and upper search bounds for `sigma`.
2. Search for `sigma` with `optimize_pdf()`.
3. Save the fitted parameters and convolution settings in a JSON file.

The helper uses a bounded one-dimensional search (Brent's method) between `1e-4` radians and `max_sigma = maximum_speed × Δt × adjustment_factor / earth_radius`. Here, Δt is derived from the emission time coordinate. It is independent of the video frame-sampling parameter `time_step` in Section 9. This bound constrains the search for `sigma`; it is not a hard speed limit on every possible movement.

The printed `f(x)` values are objective-function evaluations at trial values of `sigma`. They need not decrease monotonically as the algorithm explores the interval. Check the termination message, any warnings, and whether the fitted value lies near a search bound. Convergence within the requested tolerance does not guarantee a unique or global minimum. The appendix provides ways to compare alternative values of `sigma`.

In [ ]:
from pangeo_fish.helpers import optimize_pdf

In [ ]:
# Open the distributions
emission = xr.open_dataset(
    f"{target_root}/combined.zarr",
    engine="zarr",
    chunks=default_chunk_dims,
    inline_array=True,
    storage_options=storage_options,
)
emission

In [ ]:
# Define the parameter's bounds and search for the best value
params = optimize_pdf(
    ds=emission,
    earth_radius=earth_radius,
    adjustment_factor=adjustment_factor,
    truncate=truncate,
    maximum_speed=maximum_speed,
    tolerance=tolerance,
    dims=dims,
    # the results can be directly saved
    save_parameters=True,
    storage_options=storage_options,
    target_root=f"{target_root}",
    conv_method="LargeHealpixConv",  # fast multi-resolution convolution; "HealpixConv" is the slower reference method
)
params

## 8. State probabilities and trajectories

Using the fitted movement parameters and combined emission distribution, we calculate the fish's spatial state probabilities and derive the requested mean and mode trajectories.

In [ ]:
from pangeo_fish.helpers import predict_positions

In [ ]:
states, trajectories = predict_positions(
    target_root=f"{target_root}",
    storage_options=storage_options,
    chunks=default_chunk_dims,
    track_modes=track_modes,
    additional_track_quantities=additional_track_quantities,
    save=True,
)

Check both the spatial sum and the number of non-missing cells again. The `states` probabilities should sum to approximately 1 at each time step, and their valid-cell counts should stay above zero. Counts include zero probabilities. Other fields, such as `sigma` when present, are model parameters rather than normalized probabilities.

In [ ]:
(
    states.sum(dims).hvplot(width=500, ylim=(0, 2), title="Spatial sums by variable")
    + states.count(dims).hvplot(width=500, title="Number of non-missing grid cells")
).opts(shared_axes=False)

## 9. Visualization

We inspect the mean and mode trajectories, then animate the emission and state distributions. The emission distribution summarizes the observational evidence; the state probabilities also account for the fitted movement model. These views help interpret the estimated locations and their uncertainty over time.

### 9.1 Plotting the trajectories 

In [ ]:
from pangeo_fish.helpers import plot_trajectories

plot_trajectories(
    target_root=f"{target_root}",
    track_modes=track_modes,
    storage_options=storage_options,
    save_html=True,
)

### 9.2 Plotting the `states` and `emission` distributions 

In [ ]:
from pangeo_fish.helpers import open_distributions, render_distributions

In [ ]:
data = open_distributions(
    target_root=target_root,
    storage_options=storage_options,
    chunks=default_chunk_dims,
    chunk_time=chunk_time,
)
data

We can visualize the spatial probability distributions over time by rendering an animation and exporting it as an MP4 video. The preceding output is an xarray Dataset summary. The next cell prepares the two-dimensional coordinates required by the renderer.

`time_step = 1` requests every data time step; `3` samples every third time step. This controls frame sampling only, not the physical time interval of the movement model. **In the workshop library (`3c50042`), the renderer excludes the final time step internally**, even when `time_step = 1`.

The code cell immediately after rendering embeds the generated MP4 in the notebook. Use this inline player to view the fish animation: the JupyterLab file viewer can sometimes display an unrelated image.

In [ ]:
import numpy as np

lon = data["longitude"].values  # (nx,)
lat = data["latitude"].values  # (ny,)
lon_2d, lat_2d = np.meshgrid(lon, lat)  # (ny, nx), two-dimensional geographic coordinates

data_2d = data.rename({"latitude": "y", "longitude": "x"})
data_2d = data_2d.assign_coords(
    longitude=(("y", "x"), lon_2d),
    latitude=(("y", "x"), lat_2d),
)
data_2d

In [ ]:
video_filename = render_distributions(
    data=data_2d,
    output_path=f"{target_root}/states",
    xlim=bbox["longitude"],
    ylim=bbox["latitude"],
    time_step=time_step,
    extension="mp4",
    frames_dir="video",
    remove_frames=True,
    storage_options=storage_options,
)

In [ ]:
from IPython.display import Video, display

display(Video(filename=video_filename, embed=True))

## Appendix. Compare alternative movement parameters

To explore other values of `sigma`, use `create_parameters()` to save a chosen value or a sequence of values for different time intervals. Store each comparison in a separate output folder, and use a supported convolution class. The cells below compare the optimized result with two manually specified alternatives.

In [ ]:
import json
import warnings
from pathlib import Path
import fsspec
from pangeo_fish.helpers import create_parameters, test_parameter
from pangeo_fish.helpers import optimize_pdf

First, copy the combined emission distribution into a separate output folder.

In [ ]:
# Open the distributions
emission = xr.open_dataset(
    f"{target_root}/combined.zarr",
    engine="zarr",
    chunks=default_chunk_dims,
    inline_array=True,
    storage_options=storage_options,
)
# Create a new folder
emission.to_zarr(
    f"{target_root}/hand_sigma/combined.zarr",  # replace "hand_sigma" by another name
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

Save the chosen movement parameters in the same folder.

In [ ]:
params = create_parameters(
    sigma=[0.002708796991645453],
    sigma_indices=None,
    class_name="UpDownGaussian1DHealpix",  # Convolution class for the movement model.
    target_root=f"{target_root}/hand_sigma",  # Match the folder used above.
    save_parameters=True,
)

Alternatively, `test_parameter()` saves the combined emission distribution and the chosen parameters in a comparison folder. Pass an existing dataset as `emission`, or leave it as `None` to load `combined.zarr` from Section 6.

In [ ]:
params = test_parameter(
    emission=None,
    sigma_tested=[0.001],
    sigma_indices=None,
    Conv_method="UpDownGaussian1DHealpix",
    target_root=target_root,
    saving_root="/hand_sigma_UDG1D",
    default_chunk_dims=default_chunk_dims,
    storage_options=storage_options,
)

In [ ]:
from pangeo_fish.helpers import predict_positions

states_list = [
    predict_positions(
        target_root=f"{target_root}/{subfolder}",
        storage_options=storage_options,
        chunks=default_chunk_dims,
        track_modes=track_modes,
        additional_track_quantities=additional_track_quantities,
        save=True,
    )[0]
    for subfolder in [
        "",
        "hand_sigma",
        "hand_sigma_UDG1D",
    ]
]

Compare the state distributions using synchronized maps, then compare the trajectories below.

In [ ]:
from pangeo_fish.helpers import multi_map_with_synced_sliders, multiplot_healpix

liste = multiplot_healpix(
    [
        (states_list[0], ["states"]),
        (states_list[1], ["states"]),
        (states_list[2], ["states"]),
    ],
    refinement_level=refinement_level,
)
liste

multi_map_with_synced_sliders(liste)

In [ ]:
from pangeo_fish.helpers import plot_trajectories

traj_plots = [
    plot_trajectories(
        target_root=f"{target_root}/{subfolder}",
        track_modes=track_modes,
        storage_options=storage_options,
        save_html=True,
    ).options(title=f"Model's Folder: {subfolder}")
    for subfolder in [
        "",
        "hand_sigma",
        "hand_sigma_UDG1D",
    ]
]
(traj_plots[0] + traj_plots[1] + traj_plots[2]).cols(2)